# Plot eval_geo_mean and eval_hard_geo_mean from log.out

- x-axis: `eval_geo_mean`
- y-axis: `eval_hard_geo_mean`

Set `LOG_PATH` below, then run all cells.


In [ ]:
import ast
import os
import re
from collections import OrderedDict

import matplotlib.pyplot as plt
import numpy as np

# Example:
# LOG_PATH = "/mnt/nushare2/data/baliao/dpc/v2-base-optim_clean-final_fp32/fold0/ep20bs2x16lr1e-4/log.out"
LOG_PATH = ""


In [ ]:
if not LOG_PATH:
    raise ValueError("Please set LOG_PATH to a log.out file path")

if not os.path.isfile(LOG_PATH):
    raise FileNotFoundError(f"File not found: {LOG_PATH}")

rows = []
with open(LOG_PATH, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        if "eval_geo_mean" not in line or "eval_hard_geo_mean" not in line:
            continue
        m = re.search(r"\{.*\}", line)
        if not m:
            continue
        try:
            rec = ast.literal_eval(m.group(0))
            epoch = float(str(rec.get("epoch")))
            eval_geo = float(str(rec.get("eval_geo_mean")))
            eval_hard_geo = float(str(rec.get("eval_hard_geo_mean")))
        except Exception:
            continue
        rows.append((epoch, eval_geo, eval_hard_geo))

if not rows:
    raise ValueError("No valid metric records found in log.out")

# Keep last record per epoch in case of duplicates.
by_epoch = OrderedDict()
for epoch, eval_geo, eval_hard_geo in rows:
    by_epoch[epoch] = (eval_geo, eval_hard_geo)

epochs = sorted(by_epoch.keys())
eval_geo_vals = [by_epoch[e][0] for e in epochs]
eval_hard_geo_vals = [by_epoch[e][1] for e in epochs]

print(f"Loaded {len(epochs)} epochs from {LOG_PATH}")


In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(eval_geo_vals, eval_hard_geo_vals, s=65, alpha=0.9, label="points")

if len(eval_geo_vals) >= 2:
    x_arr = np.array(eval_geo_vals, dtype=float)
    y_arr = np.array(eval_hard_geo_vals, dtype=float)
    slope, intercept = np.polyfit(x_arr, y_arr, 1)
    x_fit = np.linspace(x_arr.min(), x_arr.max(), 100)
    y_fit = slope * x_fit + intercept
    plt.plot(x_fit, y_fit, color="crimson", linewidth=2, label=f"fit: y={slope:.3f}x+{intercept:.3f}")

for e, x, y in zip(epochs, eval_geo_vals, eval_hard_geo_vals):
    ep = int(e) if abs(e - round(e)) < 1e-8 else e
    plt.annotate(str(ep), (x, y), fontsize=8, xytext=(4, 4), textcoords="offset points")

plt.xlabel("eval_geo_mean")
plt.ylabel("eval_hard_geo_mean")
plt.title("eval_hard_geo_mean vs eval_geo_mean")
plt.grid(alpha=0.25, linestyle="--")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Optional: quick summary
best_i = max(range(len(eval_geo_vals)), key=lambda i: eval_geo_vals[i])
best_hard_i = max(range(len(eval_hard_geo_vals)), key=lambda i: eval_hard_geo_vals[i])
print(
    f"Best eval_geo_mean: {eval_geo_vals[best_i]:.4f} at epoch {epochs[best_i]} | "
    f"hard@same_epoch: {eval_hard_geo_vals[best_i]:.4f}"
)
print(
    f"Best eval_hard_geo_mean: {eval_hard_geo_vals[best_hard_i]:.4f} at epoch {epochs[best_hard_i]} | "
    f"geo@same_epoch: {eval_geo_vals[best_hard_i]:.4f}"
)
